In [ ]:
import json
import math
import os
from collections.abc import Callable, Sequence
from typing import Any, Dict, Literal, Optional, Type, Union

import cv2
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils
import torch
import torch.nn.functional as F
import torchvision.transforms.v2 as T
import torchvision.transforms.v2.functional as TF
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import Dataset
from torchvision import tv_tensors
from torchvision.io import read_image
from torchvision.ops import masks_to_boxes
from torchvision.transforms import InterpolationMode
from tqdm.notebook import tqdm

root = rootutils.setup_root(search_from=".", indicator=".project-root", pythonpath=True)

from src.data.components.dataset import (
    FetalBrainPlanesDataset,
    HeadSegmentationDataset,
    USVideosFrameDataset,
    USVideosSsimFrameDataset,
)
from src.data.components.transforms import PadToAspectRatio, Resize
from src.data.utils.utils import (
    read_image_tensor,
    save_image_tensor,
    show_numpy_images,
    show_pytorch_images,
)
from src.models.head_segmentation_module import HeadSegmentationLitModule

data_dir = root / "data"
dataset_name = "FETAL_HEAD_SEGMENTATION"
dataset_dir = data_dir / dataset_name

In [ ]:
%matplotlib ipympl

import matplotlib

matplotlib.use("ipympl")

# Prepare FETAL_HEAD_SEGMENTATION dataset

## Prepare data

### Prepare segmentation masks

In [ ]:
def prepare_segmentation_masks(folder):
    data_path = dataset_dir / folder
    images_path = data_path / "Images"
    segmentations_path = data_path / "SegmentationClass"

    images = list(images_path.iterdir())
    for image_path in tqdm(images, desc="Prepare images"):

        segmentation_path = segmentations_path / image_path.name
        segmentation = np.array(Image.open(segmentation_path))

        red_color = [255, 0, 0]
        green_color = [0, 255, 0]
        blue_color = [0, 0, 255]

        if np.any((segmentation == red_color).all(axis=-1)):

            mask = (
                (segmentation == red_color).all(axis=-1)
                | (segmentation == green_color).all(axis=-1)
                | (segmentation == blue_color).all(axis=-1)
            )
            segmentation[mask] = red_color

            segmentation = Image.fromarray(segmentation)
            segmentation_path = data_path / "Segmentations" / segmentation_path.name
            # segmentation.save(segmentation_path)
        else:
            unselected_images.append((images_path / image_path.name, segmentations_path / image_path.name))


unselected_images = []
prepare_segmentation_masks(folder="Trans-cerebellum")
prepare_segmentation_masks(folder="Trans-thalamic")
prepare_segmentation_masks(folder="Trans-ventricular")
prepare_segmentation_masks(folder="HC18")

In [ ]:
images = []
for image_path, segmentation_path in unselected_images:
    if image_path.exists():
        image = read_image(image_path)
        image = TF.to_grayscale(image)
        images.append((image, f"{image_path.stem}"))

    if segmentation_path.exists():
        image = read_image(segmentation_path)
        images.append((image))

if len(images) > 0:
    show_pytorch_images(images).show()

### Fill missing FETAL_PLANES segmentation masks

#### Read CSV

In [ ]:
fix_path = dataset_dir / "FETAL_PLANES_SEGMENTATION_FIX.json"
with open(fix_path, "r") as fix_file:
    fix_data = json.load(fix_file)

extracted_data = []
for image_info in fix_data.values():
    filename = image_info.get("filename")
    regions = image_info.get("regions", [])

    if not regions:
        print(filename)

    for region in regions:
        shape_attributes = region.get("shape_attributes", {})

        if shape_attributes.get("name") == "ellipse":
            ellipse_data = {
                "filename": filename,
                "cx": shape_attributes.get("cx"),
                "cy": shape_attributes.get("cy"),
                "rx": shape_attributes.get("rx"),
                "ry": shape_attributes.get("ry"),
                "theta": shape_attributes.get("theta"),
            }
            extracted_data.append(ellipse_data)

len(pd.DataFrame(extracted_data))

#### Generate masks

In [ ]:
def create_ellipse_tensor(
    height: int,
    width: int,
    center_x: int,
    center_y: int,
    radius_x: int,
    radius_y: int,
    theta_rad: float,
) -> torch.Tensor:
    """
    Creates a 2D PyTorch tensor with a binary ellipse shape.

    Args:
        height (int): The height of the tensor.
        width (int): The width of the tensor.
        center_x (int): The x-coordinate of the ellipse's center.
        center_y (int): The y-coordinate of the ellipse's center.
        radius_x (int): The radius of the ellipse along the x-axis.
        radius_y (int): The radius of the ellipse along the y-axis.
        theta_rad (float): The rotation angle in radians.

    Returns:
        torch.Tensor: A 2D tensor of shape (height, width) with 1s inside the
                      ellipse and 0s outside.
    """
    # Create a grid of coordinates
    y, x = torch.meshgrid(torch.arange(height), torch.arange(width), indexing="ij")

    # Translate the coordinates so the center of the ellipse is at (0, 0)
    x_translated = x - center_x
    y_translated = y - center_y

    # Pre-calculate the trigonometric values for the rotation
    cos_theta = math.cos(theta_rad)
    sin_theta = math.sin(theta_rad)

    # Apply the rotation transformation to the coordinates
    x_rotated = x_translated * cos_theta + y_translated * sin_theta
    y_rotated = -x_translated * sin_theta + y_translated * cos_theta

    # Apply the ellipse equation to the rotated coordinates
    ellipse_mask = (x_rotated / radius_x) ** 2 + (y_rotated / radius_y) ** 2 <= 1

    # Convert the boolean mask to a float tensor
    return ellipse_mask.float()


fetal_planes = pd.read_csv(f"{data_dir}/FETAL_PLANES/data_fix.csv")

for ellipse_data in tqdm(extracted_data):
    image_name = ellipse_data["filename"]
    cx = ellipse_data["cx"]
    cy = ellipse_data["cy"]
    rx = ellipse_data["rx"]
    ry = ellipse_data["ry"]
    theta = ellipse_data["theta"]

    if (dataset_dir / "Fix" / "Segmentations" / image_name).exists():
        continue

    image_path = data_dir / "FETAL_PLANES" / "Images" / image_name
    image = read_image_tensor(image_path)

    mask = create_ellipse_tensor(image.shape[1], image.shape[2], cx, cy, rx, ry, theta)
    mask = mask.unsqueeze(0)
    mask = mask.repeat(3, 1, 1)
    mask = mask * 255
    mask = torch.clamp(mask, min=0, max=255)
    mask = mask.to(torch.uint8)

    image_path = dataset_dir / "Fix" / "Images" / image_name
    save_image_tensor(image, image_path)

    mask_path = dataset_dir / "Fix" / "Segmentations" / image_name
    save_image_tensor(mask, mask_path)

### Prepare segmentation masks JNU-IFM (skip due to love quality)

In [ ]:
def prepare_segmentation_masks():
    data_path = data_dir / "JNU-IFM" / "us_data"

    data = []
    videos = list(data_path.iterdir())
    for video_id, video_path in enumerate(tqdm(videos, desc="Prepare videos")):
        images_path = video_path / "image"
        segmentations_path = video_path / "mask_enhance"

        images = list(images_path.iterdir())
        for image_path in tqdm(images, desc="Prepare images", leave=False):

            image = Image.open(image_path)
            image_path = dataset_dir / "JNU-IFM" / "Images" / image_path.name
            image.save(image_path)

            segmentation_path = segmentations_path / f"{image_path.stem}_mask.png"
            segmentation = np.array(Image.open(segmentation_path))

            black_color = [0, 0, 0]
            red_color = [255, 0, 0]
            green_color = [0, 255, 0]
            blue_color = [0, 0, 255]

            brain_plane = 1 if np.any((segmentation == green_color).all(axis=-1)) else 0

            segmentation[(segmentation == red_color).all(axis=-1)] = black_color
            segmentation[(segmentation == green_color).all(axis=-1)] = red_color

            segmentation = Image.fromarray(segmentation)
            segmentation_path = dataset_dir / "JNU-IFM" / "Segmentations" / image_path.name
            segmentation.save(segmentation_path)

            data.append(
                {
                    "Image_name": image_path.stem,
                    "Patient_num": video_id,
                    "Frame_num": int(image_path.stem.replace(video_path.name + "_", "")),
                    "Brain_plane": brain_plane,
                    "Ultrasound_path": str(image_path.relative_to(dataset_dir)),
                    "Segmentation_path": str(segmentation_path.relative_to(dataset_dir)),
                }
            )

    return data


data = prepare_segmentation_masks()
data_df = pd.DataFrame(data)
data_df = data_df.sort_values(["Patient_num", "Frame_num"])
data_df = data_df.reset_index(drop=True)
data_df.to_csv(dataset_dir / "JNU-IFM" / "data.csv", index=False)
data_df

### Prepare segmentation masks PSFHS (skip)

In [ ]:
# import SimpleITK as sitk


# images_path = data_dir / "PSFHS" / "image_mha"
# segmentations_path = data_dir / "PSFHS" / "label_mha"

# images_ = []

# images = list(images_path.iterdir())
# for image_path in tqdm(images, desc="Prepare images", leave=False):
#     mask_path = segmentations_path / image_path.name

#     image_sitk = sitk.ReadImage(image_file)
#     mask_sitk = sitk.ReadImage(mask_path, sitk.sitkUInt8)

#     image_array = np.transpose(sitk.GetArrayFromImage(image_sitk), (1, 2, 0))
#     mask_array = sitk.GetArrayFromImage(mask_sitk)
#     images_.extend([image_array, mask_array])

#     if len(images_) >= 1000:
#         break

# show_numpy_images(images_[:64])

## Prepare csv

### Read all images with mask

In [ ]:
def get_segmentation_masks_data(folder, brain_plane):
    data_path = dataset_dir / folder
    images_path = data_path / "Images"
    segmentations_path = data_path / "Segmentations"

    images = list(images_path.iterdir())
    for image_path in tqdm(images, desc="Prepare images"):

        segmentation_path = segmentations_path / image_path.name

        if segmentation_path.exists():
            data.append(
                {
                    "Image_name": image_path.stem,
                    "Patient_num": None,
                    "Brain_plane": 1,
                    "Plane": brain_plane,
                    "Ultrasound_path": str(image_path.relative_to(dataset_dir)),
                    "Segmentation_path": str(segmentation_path.relative_to(dataset_dir)),
                }
            )
        # break


data = []
get_segmentation_masks_data(
    folder="Trans-cerebellum",
    brain_plane="Trans-cerebellum",
)
get_segmentation_masks_data(
    folder="Trans-thalamic",
    brain_plane="Trans-thalamic",
)
get_segmentation_masks_data(
    folder="Trans-ventricular",
    brain_plane="Trans-ventricular",
)
get_segmentation_masks_data(
    folder="Fix",
    brain_plane="Other",
)
get_segmentation_masks_data(
    folder="HC18",
    brain_plane="Other",
)

print(len(data))
data_df = pd.DataFrame(data)
data_df

### Load data

In [ ]:
fetal_planes = pd.read_csv(f"{data_dir}/FETAL_PLANES/data_fix.csv")
fetal_planes

### Add not a brain planes

Add not a brain images from FETAL_PLANES, with the same Train value as defined in FETAL_PLANES

In [ ]:
fetal_not_a_brain_planes = fetal_planes[fetal_planes["Brain_plane"] == "Not A Brain"]
for _, row in tqdm(fetal_not_a_brain_planes.iterrows(), total=len(fetal_not_a_brain_planes)):

    existing_index = data_df[data_df["Image_name"] == row["Image_name"]].index.to_list()
    if existing_index:
        index = existing_index[0]
    else:
        index = len(data_df)

    data_df.loc[index] = {
        "Image_name": row["Image_name"],
        "Brain_plane": 0,
        "Plane": "Not A Brain",
        "Ultrasound_path": f"FETAL_PLANES/Images/{row["Image_name"]}.png",
        "Segmentation_path": None,
    }


print(len(data_df))
data_df
len(data_df[data_df["Brain_plane"] == 1])

### Assign FETAL_PLANES value

In [ ]:
data_df["Patient_num"] = None
data_df["Valid"] = 1
new_brain_images = 0

for _, row in tqdm(fetal_planes.iterrows(), total=len(fetal_planes)):
    for index, _ in data_df[data_df["Image_name"] == row["Image_name"]].iterrows():
        data_df.loc[index, "Patient_num"] = str(row["Patient_num"])
        data_df.loc[index, "Valid"] = 1 if (pd.isna(row["Duplicate"]) and row["Valid"] == 1) else 0
        if isinstance(row["New_brain_plane"], str):
            data_df.loc[index, "Valid"] = 1
            data_df.loc[index, "Brain_plane"] = 1
            data_df.loc[index, "Segmentation_path"] = f"Fix/Segmentations/{data_df.Image_name[index]}.png"
            new_brain_images += 1

print(f"Images:           {len(data_df)}")
print(f"Brain images:     {len(data_df[data_df["Brain_plane"] == 1])}")
print(f"New brain images: {new_brain_images}")

### Add JNU-IFM images (skip)

In [ ]:
jnu_df = pd.read_csv(dataset_dir / "JNU-IFM" / "data.csv")

for _, row in tqdm(jnu_df.iterrows(), total=len(jnu_df)):

    existing_index = data_df[data_df["Image_name"] == row["Image_name"]].index.to_list()
    if existing_index:
        index = existing_index[0]
    else:
        index = len(data_df)

    data_df.loc[index] = {
        "Image_name": row["Image_name"],
        "Patient_num": str(row["Patient_num"] + 10000),
        "Brain_plane": row["Brain_plane"],
        "Plane": "Other" if row["Brain_plane"] == 1 else "Not A Brain",
        "Ultrasound_path": row["Ultrasound_path"],
        "Segmentation_path": row["Segmentation_path"] if row["Brain_plane"] == 1 else None,
    }


print(len(data_df))
print(len(data_df[data_df["Brain_plane"] == 1]))

### Assign Train and Subset value 

For images from FETAL_PLANES, Train value will be the same as defined in FETAL_PLANES.  
For images from HC18, Train value will be 1
(the same as for Brain Planes experiments)

In [ ]:
data_df["Train"] = 1
data_df["Subset"] = "train"

for _, row in tqdm(fetal_planes.iterrows(), total=len(fetal_planes)):
    for index, _ in data_df[data_df["Image_name"] == row["Image_name"]].iterrows():
        data_df.loc[index, "Train"] = row["Train"]
        data_df.loc[index, "Subset"] = row["Subset"]

### Statistics

In [ ]:
data_df["Subset"].value_counts()

In [ ]:
data_df["Brain_plane"].value_counts()

In [ ]:
df = data_df[data_df["Brain_plane"] == 1]
df = df[df["Valid"] == 1]
df["Subset"].value_counts()

### Sort dataset

In [ ]:
data_df = data_df.sort_values("Image_name")

### Save csv

In [ ]:
data_df.to_csv(dataset_dir / "data.csv", index=False)
data_df

## Split Data

### Test seed

In [ ]:
data_df = pd.read_csv(dataset_dir / "data.csv", dtype={"Patient_num": str})
data_df = data_df[data_df["Valid"] == 1]
data_df = data_df[data_df["Patient_num"].notna()]
data_df

In [ ]:
def group_split_label(
    dataset: pd.DataFrame, test_size: float, groups, random_state: int = None
) -> tuple[pd.DataFrame, pd.DataFrame]:
    splitter = GroupShuffleSplit(test_size=test_size, n_splits=1, random_state=random_state)
    split = splitter.split(dataset, groups=groups)
    train_idx, test_idx = next(split)
    return dataset.iloc[train_idx].reset_index(drop=True), dataset.iloc[test_idx].reset_index(drop=True)


def get_similarity(train, test, test_size):
    similarity = 0
    train_count = train.value_counts(sort=False).sort_index()
    test_count = test.value_counts(sort=False).sort_index()

    if train_count.index.tolist() != test_count.index.tolist():
        return -1

    for a, b in zip(train_count, test_count):
        similarity += (a * test_size - b * (1 - test_size)) ** 2
    return similarity**0.5


def plt_value_counts(ax, dataset, tile=None):
    counts = dataset.value_counts(sort=False).sort_index()
    counts.plot(kind="bar", ax=ax)
    if tile:
        ax.set_title(tile)


def plt_group_split(dataset: pd.DataFrame, test_size: float, random_states: list[int], top_states: int = None):
    splits = []
    for random_state in tqdm(random_states):
        tr, val = group_split_label(
            dataset,
            test_size=test_size,
            groups=dataset["Patient_num"],
            random_state=random_state,
        )

        similarity = get_similarity(tr.Brain_plane, val.Brain_plane, test_size)
        if similarity >= 0:
            splits.append((similarity, tr, val, random_state))

    splits.sort(key=lambda e: (e[0], e[3]))
    nrows = top_states if top_states else len(splits)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=2,
        sharex="all",
        squeeze=False,
        figsize=(20, 3 * nrows),
    )
    fig.suptitle(f"Test size {test_size}")
    for i, (similarity, tr, val, random_state) in enumerate(splits[:nrows]):
        plt_value_counts(axes[i, 0], tr.Brain_plane, tile=f"Seed {random_state}")
        plt_value_counts(axes[i, 1], val.Brain_plane, tile=f"Similarity {similarity}")

    plt.show()
    print([random_state for (similarity, tr, val, random_state) in splits[:nrows]])


plt_group_split(
    data_df,
    test_size=0.3,
    random_states=list(range(10000)),
    top_states=10,
)  # [7312, 2481, 3290, 7276, 2340, 9292, 1871, 7795, 6620, 5542]

In [ ]:
data_train, data_test = group_split_label(
    data_df,
    test_size=0.3,
    groups=data_df["Patient_num"],
    random_state=2340,
)

plt_group_split(
    data_train,
    test_size=0.2,
    random_states=list(range(10000)),
    top_states=10,
)
# 7312 -> 782, 3773, 5264, 9396, 2895
# 2481 -> 1078, 7503, 1474, 3427, 5375
# 3290 -> 391, 5289, 6328, 1020, 3168
# 7276 -> 6904, 2286, 7276, 1691, 1157
# 2340 -> 4261, 1993, 336, 1574, 6101

### Split

In [ ]:
import random


def random_seeds(data):
    random_row = random.choice(data)
    seed_1 = random_row[0]
    seed_2 = random.choice(random_row[1])
    return seed_1, seed_2


def split_dataset(df, seed_1, seed_2):
    train_df, test_df = group_split_label(
        df,
        test_size=0.4,
        groups=df["Patient_num"],
        random_state=seed_1,
    )
    train_df = train_df.reset_index(drop=True)

    train_df, val_df = group_split_label(
        train_df,
        test_size=0.2,
        groups=train_df["Patient_num"],
        random_state=seed_2,
    )
    return train_df, val_df, test_df


data_df = pd.read_csv(dataset_dir / "data.csv", dtype={"Patient_num": str})
data_df = data_df[data_df["Valid"] == 1]
data_df = data_df[data_df["Patient_num"].notna()]

seeds = [
    [7312, [782, 3773, 5264, 9396, 2895]],
    [2481, [1078, 7503, 1474, 3427, 5375]],
    [3290, [391, 5289, 6328, 1020, 3168]],
    [7276, [6904, 2286, 7276, 1691, 1157]],
    [2340, [4261, 1993, 336, 1574, 6101]],
]
seed_1, seed_2 = random_seeds(seeds)
train_df, val_df, test_df = split_dataset(data_df, seed_1, seed_2)

data_df = pd.read_csv(dataset_dir / "data.csv", dtype={"Patient_num": str})
data_df["Subset"] = "train"
for index, row in tqdm(data_df.iterrows(), total=len(data_df)):
    for _ in train_df[train_df["Image_name"] == row["Image_name"]].iterrows():
        data_df.loc[index, "Subset"] = "train"
    for _ in val_df[val_df["Image_name"] == row["Image_name"]].iterrows():
        data_df.loc[index, "Subset"] = "val"
    for _ in test_df[test_df["Image_name"] == row["Image_name"]].iterrows():
        data_df.loc[index, "Subset"] = "test"


df = data_df[data_df["Brain_plane"] == 1]
df = df[df["Valid"] == 1]
df["Subset"].value_counts()

Subset
train    2298
test     1450
val       339

In [ ]:
data_df.to_csv(dataset_dir / "data.csv", index=False)
data_df

# Visualize Data

In [ ]:
data_df = pd.read_csv(dataset_dir / "data.csv", dtype={"Patient_num": str})
fetal_planes = pd.read_csv(f"{data_dir}/FETAL_PLANES/data_fix.csv")
predictions_df = pd.read_csv(dataset_dir / "prediction.csv")
len(predictions_df)

## Unidentified brain images

In [ ]:
misidentified_images = predictions_df[predictions_df["Prediction"] == 0]
misidentified_images = misidentified_images.sort_values(["Count", "Image_name"], ascending=[False, True])
print(len(misidentified_images))

image_size = (192, 256)
pad_to_aspect_ration = PadToAspectRatio(image_size, fill=0)

images = []
ignore = [
    # Ignore Not a Brain images
    # numbers
    # Ignore Brain images
    # ommited for now
]

for _, misidentified_row in tqdm(misidentified_images.iterrows(), total=len(misidentified_images)):
    for index, row in fetal_planes[fetal_planes["Image_name"] == misidentified_row["Image_name"]].iterrows():
        for _, data_row in data_df[data_df["Image_name"] == misidentified_row["Image_name"]].iterrows():
            if index in ignore or data_row["Brain_plane"] == 0:
                continue
            image_name = row["Image_name"]
            plane = row["Plane"]
            brain_plane = row["Brain_plane"]
            subset = row["Subset"]

            image_path = f"{data_dir}/FETAL_PLANES/Images/{image_name}.png"
            image = read_image_tensor(image_path)
            image = pad_to_aspect_ration(image)
            images.append((image, f"{index}: {brain_plane} ({misidentified_row["Count"]})"))
            # print(f"{index:<6}: {image_name:<30} {brain_plane:<20} {misidentified_row["Count"]}")

print(len(images))
if images:
    show_pytorch_images(images[:25], gray=False).show()

## Not a brain images identify as brain image

In [ ]:
misidentified_images = predictions_df[predictions_df["Prediction"] == 1]
misidentified_images = misidentified_images.sort_values(["Count", "Image_name"], ascending=[False, True])
print(len(misidentified_images))

image_size = (192, 256)
pad_to_aspect_ration = PadToAspectRatio(image_size, fill=0)

images = []
ignore = [
    # Ignore Not a Brain images
    # Ignore Brain images
    # ommited for now
]

limit = 16

for _, misidentified_row in tqdm(misidentified_images.iterrows(), total=len(misidentified_images)):
    for index, row in fetal_planes[fetal_planes["Image_name"] == misidentified_row["Image_name"]].iterrows():
        for _, data_row in data_df[data_df["Image_name"] == misidentified_row["Image_name"]].iterrows():
            if index in ignore or data_row["Brain_plane"] == 1:
                continue
            image_name = row["Image_name"]
            plane = row["Plane"]
            brain_plane = row["Brain_plane"]
            subset = row["Subset"]

            image_path = f"{data_dir}/FETAL_PLANES/Images/{image_name}.png"
            image = read_image_tensor(image_path)
            image = pad_to_aspect_ration(image)
            # images.append((image, f"{index}: {plane} ({misidentified_row["Count"]})"))
            images.append((image, f"{index}: ({misidentified_row["Count"]})"))
            # print(f"{index}, ")
            print(f"{index:<6}: {image_name:<30} {plane:<18} {misidentified_row["Count"]}")

            if len(images) >= limit:
                break
        if len(images) >= limit:
            break
    if len(images) >= limit:
        break

print(len(images))
if images:
    show_pytorch_images(images[:limit], gray=False).show()

In [ ]:
def display_image(img, ax, title):
    img = img.detach()
    img = TF.to_pil_image(img)
    img = TF.to_grayscale(img)
    ax.imshow(np.asarray(img), cmap="gray")
    ax.set_title(title)
    ax.axis("off")


def overlap_mask(image: torch.Tensor, mask: torch.Tensor, ax, title):
    image = image.clone()
    image = TF.to_grayscale(image)
    if image.shape[0] == 1:
        image = image.repeat(3, 1, 1)
    if image.max() <= 1.0:
        image = image * 255

    image = image.int()
    image[0] = image[0] + mask * 255 * 0.4
    image = torch.clamp(image, min=0, max=255)
    image = image.to(torch.uint8)

    image = TF.to_image(image)
    image = image.permute(1, 2, 0).numpy()

    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")


def display_image_and_mask(image, mask):
    fig, axes = plt.subplots(ncols=3, nrows=1, squeeze=False, figsize=(15, 4))
    mask = mask.float()
    display_image(image, axes[0, 0], "Image")
    display_image(mask, axes[0, 1], "Mask")
    overlap_mask(image, mask, axes[0, 2], "Overlayed Mask")

    plt.tight_layout()
    plt.show()

    return


input_size = (165, 240)
transform = T.Compose(
    [
        T.Grayscale(),
        # RandomPercentCrop(max_percent=20),
        T.Resize(input_size, interpolation=InterpolationMode.NEAREST),
        # T.AutoAugment(T.AutoAugmentPolicy.IMAGENET),
        # T.RandAugment(magnitude=11),
        # T.TrivialAugmentWide(),
        # T.AugMix(),
        # T.RandomHorizontalFlip(p=0.5),
        # T.RandomVerticalFlip(p=0.5),
        # T.RandomAffine(degrees=20, translate=(0.1, 0.1), scale=(1.0, 1.2)),
        T.ToDtype(dtype={tv_tensors.Image: torch.float32, tv_tensors.Mask: torch.float32, "others": None}, scale=True),
        # T.Normalize(mean=[0.17], std=[0.19]),  # FetalBrain
        # T.Normalize(mean=[0.449], std=[0.226]),  # ImageNet
    ]
)

dataset = HeadSegmentationDataset(
    data_dir=data_dir,
    subset="train",
    transform=None,
)

print(len(dataset))
image, mask, label = dataset[2013]
print("shape")
print(image.shape)
print(mask.shape)
print(label.shape)
print("values")
print(image.min(), image.max())
print(torch.unique(mask))
print(label)
print("")

dataset = HeadSegmentationDataset(
    data_dir=data_dir,
    subset="train",
    transform=transform,
)

image_t, mask_t, label_t = dataset[2046]
print("shape")
print(image_t.shape)
print(mask_t.shape)
print(label_t.shape)
print("values")
print(image_t.min(), image_t.max())
print(torch.unique(mask_t))
print(label_t)
print("types")
print(mask_t.dtype)
print(label_t.dtype)
print("")

image_t2, mask_t2, label_t2 = dataset[6552]
print("shape")
print(image_t2.shape)
print(mask_t2.shape)
print(label_t2.shape)
print("values")
print(image_t2.min(), image_t2.max())
print(torch.unique(mask_t2))
print(label_t2)
print("types")
print(mask_t2.dtype)
print(label_t2.dtype)


display_image_and_mask(image, mask)
display_image_and_mask(image_t, mask_t)
display_image_and_mask(image_t2, mask_t2)
# show_pytorch_images([image, mask, image_t, mask_t]).show()

In [ ]:
ds = dataset.labels
ds = ds[ds["Patient_num"].notna()]
ds = ds[ds["Brain_plane"] == 1]
ds

In [ ]:
dataset = HeadSegmentationDataset(
    data_dir=data_dir,
    subset="train",
    transform=T.Compose(
        [
            T.Grayscale(),
            PadToAspectRatio(
                (192, 256),
                fill={tv_tensors.Image: 0, tv_tensors.Mask: 0, "others": None},
            ),
            Resize(
                (192, 256),
                interpolation={
                    tv_tensors.Image: T.InterpolationMode.BILINEAR,
                    tv_tensors.Mask: T.InterpolationMode.NEAREST_EXACT,
                    "others": None,
                },
                # interpolation={tv_tensors.Image: "bilinear", tv_tensors.Mask: "nearest-exact", "others": None},
            ),
            T.ToDtype(
                dtype={tv_tensors.Image: torch.float32, tv_tensors.Mask: torch.float32, "others": None}, scale=True
            ),
        ]
    ),
)

image = dataset[0, 0]
image, mask, label = dataset[0]
print("shape")
print(image.shape)
print(mask.shape)
print(label.shape)
print("")
display_image_and_mask(image, mask)

In [ ]:
videos = USVideosSsimFrameDataset(
    data_dir=data_dir,
    dataset_name="US_VIDEOS_ssim_0.7",
    transform=torch.nn.Sequential(
        T.Grayscale(),
        T.Resize((165, 240), interpolation=T.InterpolationMode.NEAREST),
        T.ToDtype(dtype={tv_tensors.Image: torch.float32, tv_tensors.Mask: torch.float32, "others": None}, scale=True),
    ),
)

fig, ax = plt.subplots()

video = videos[1]
frame = video[0]
frame = frame.detach()
frame = TF.to_pil_image(frame)
frame = TF.to_grayscale(frame)
ax.imshow(np.asarray(frame), cmap="gray")
ax.axis("off")

frames = []
for idx in tqdm(range(len(video))):
    frame = video[idx]
    frame = frame.detach()
    frame = TF.to_pil_image(frame)
    frame = TF.to_grayscale(frame)
    frame = ax.imshow(np.asarray(frame), cmap="gray", animated=True)
    frames.append([frame])

ani = animation.ArtistAnimation(fig, frames, interval=50, blit=True, repeat_delay=1000)
plt.show()

In [ ]:
def display_image_animation(image, ax, title):
    image = image.detach()
    image = TF.to_pil_image(image)
    image = TF.to_grayscale(image)
    image = np.asarray(image)

    if not ax.images:
        ax.imshow(image, cmap="gray")
        ax.set_title(title)
        ax.axis("off")

    return ax.imshow(image, cmap="gray", animated=True)


def display_overlap_mask_animation(image: torch.Tensor, mask: torch.Tensor, ax, title):
    image = image.clone()
    image = TF.to_grayscale(image)
    if image.shape[0] == 1:
        image = image.repeat(3, 1, 1)
    if image.max() <= 1.0:
        image = image * 255

    image = image.int()
    image[0] = image[0] + mask * 255 * 0.4
    image = torch.clamp(image, min=0, max=255)
    image = image.to(torch.uint8)

    image = TF.to_image(image)
    image = image.permute(1, 2, 0).numpy()

    if not ax.images:
        ax.imshow(image)
        ax.set_title(title)
        ax.axis("off")

    return ax.imshow(image, animated=True)


checkpoint_file = root / "logs" / "head_segmentation_train" / "runs" / "2025-08-25_13-36-03"
checkpoint = sorted(checkpoint_file.glob("checkpoints/epoch_*.ckpt"))[-1]
model = HeadSegmentationLitModule.load_from_checkpoint(checkpoint_path=str(checkpoint))
# disable randomness, dropout, etc...
model.eval()
model.to("cpu")

videos = USVideosSsimFrameDataset(
    data_dir=data_dir,
    dataset_name="US_VIDEOS_ssim_0.7",
    transform=torch.nn.Sequential(
        T.Grayscale(),
        T.Resize((165, 240), interpolation=T.InterpolationMode.NEAREST),
        T.ToDtype(dtype=torch.float32, scale=True),
    ),
)

# fig, axes = plt.subplots(ncols=3, nrows=1, squeeze=False, figsize=(10, 4))

# video = videos[1]
# frames = []
# for idx in tqdm(range(len(video))):
#     frame = video[idx]
#     with torch.no_grad():
#         logits = model(frame.unsqueeze(0)).squeeze(0)
#         prediction = (torch.sigmoid(logits) > 0.5).float()

#     im1 = display_image_animation(frame, axes[0, 0], "Image")
#     im2 = display_image_animation(prediction, axes[0, 1], "Prediction")
#     im3 = display_overlap_mask_animation(frame, prediction, axes[0, 2], "Overlayed Mask")
#     frames.append([im1, im2, im3])

# ani = animation.ArtistAnimation(fig, frames, interval=100, blit=True, repeat_delay=1000)
# plt.tight_layout()
# plt.show()

In [ ]:
def predict_head_mask(model: HeadSegmentationLitModule, image: torch.Tensor) -> tuple[torch.Tensor, int]:
    """Binary mask [1, H, W] and frame label (same rules as HeadSegmentationLitModule.calculate_prediction)."""
    logits = model(image.unsqueeze(0))
    binary_mask, prediction_labels = model.calculate_hard_prediction(logits)
    return binary_mask.squeeze(0), int(prediction_labels.squeeze(0).item())


def find_angle(mask):
    mask = mask.squeeze(0)

    # Get the indices of the non-zero elements
    coords = torch.nonzero(mask, as_tuple=False).float()
    # Center the data by subtracting the mean
    mean = torch.mean(coords, dim=0)
    centered_coords = coords - mean

    # Calculate the covariance matrix
    covariance_matrix = torch.matmul(centered_coords.T, centered_coords)
    # Perform eigenvalue decomposition to find eigenvectors (principal components)
    eigenvalues, eigenvectors = torch.linalg.eigh(covariance_matrix)
    # The eigenvector corresponding to the largest eigenvalue (the first principal component)
    principal_axis = eigenvectors[:, 1]
    # Calculate the angle of the principal axis
    angle = torch.atan2(principal_axis[0], principal_axis[1]) * 180 / torch.pi

    return angle.round().int()


video = videos[1]
frame = video[0]
# frame = TF.rotate(frame, angle=120)
with torch.no_grad():
    prediction, _ = predict_head_mask(model, frame)

for base_angle in range(0, 360, 30):
    prediction2 = TF.rotate(prediction, angle=base_angle)
    angle = find_angle(prediction2)
    print(f"Base angle: {base_angle}, angle: {angle}, result: {base_angle + angle}")
# break

In [ ]:
def crop(image, x1, y1, x2, y2, pad=10):
    pad_x = int((x2 - x1) * (pad / 100.0))
    left_pad = pad_x // 2
    right_pad = pad_x - left_pad

    pad_y = int((y2 - y1) * (pad / 100.0))
    top_pad = pad_y // 2
    bottom_pad = pad_y - top_pad

    x1 = max(x1 - left_pad, 0)
    y1 = max(y1 - top_pad, 0)
    x2 = min(x2 + right_pad, image.shape[2])
    y2 = min(y2 + bottom_pad, image.shape[1])
    return TF.crop(image, y1, x1, y2 - y1, x2 - x1)


def pad_central(image, height, width):
    original_height, original_width = image.shape[1:]

    padding_needed = height - original_height
    top_pad = padding_needed // 2
    bottom_pad = padding_needed - top_pad

    padding_needed = width - original_width
    left_pad = padding_needed // 2
    right_pad = padding_needed - left_pad

    return TF.pad(image, padding=[left_pad, top_pad, right_pad, bottom_pad], fill=0)


class StabilizeVideoEma:
    def __init__(self, alpha: float = 0.5):
        self.alpha = alpha
        self.args = None
        self.int_type = False

    def stabilize(self, *args):
        if self.args is None:
            self.args = self._extract_item_from_tensor(args)

            for arg in self.args:
                if isinstance(arg, int):
                    self.int_type = True
        else:
            for i in range(len(self.args)):
                self.args[i] = self.alpha * args[i] + (1 - self.alpha) * self.args[i]

        rs = self._extract_item_from_tensor(self.args)
        rs = [int(round(arg)) if self.int_type else arg for arg in rs]

        if len(rs) == 1:
            return rs[0]
        else:
            return rs

    def _extract_item_from_tensor(self, args):
        return [args[i].item() if isinstance(args[i], torch.Tensor) else args[i] for i in range(len(args))]


videos = USVideosFrameDataset(
    data_dir=data_dir,
    train=False,
    transform=torch.nn.Sequential(
        T.Grayscale(),
        T.ToDtype(dtype=torch.float32, scale=True),
    ),
)

fig, axes = plt.subplots(ncols=3, nrows=3, squeeze=False, figsize=(10, 8))
video = videos[0]
frames = []

image_size = (165, 240)
interpolation = T.InterpolationMode.NEAREST

frame_crop_prev = None
frame_rotate_crop_prev = None

alpha = 0.5
ema_stabilizer_crop = StabilizeVideoEma(alpha=alpha)
ema_stabilizer_rotate = StabilizeVideoEma(alpha=alpha)
ema_stabilizer_rotate_crop = StabilizeVideoEma(alpha=alpha)

alpha = 0.1
ema_stabilizer_crop_2 = StabilizeVideoEma(alpha=alpha)
ema_stabilizer_rotate_2 = StabilizeVideoEma(alpha=alpha)
ema_stabilizer_rotate_crop_2 = StabilizeVideoEma(alpha=alpha)

pad = 5
image_pad_central = (700, 900)
# image_size_crop=(175, 225)
image_size_crop = (88, 112)

for idx in tqdm(range(len(video))):
    frame_base = video[idx]
    frame = TF.resize(frame_base, size=image_size, interpolation=interpolation)
    with torch.no_grad():
        prediction, _ = predict_head_mask(model, frame)
        prediction_base = TF.resize(prediction, size=frame_base.shape[1:], interpolation=interpolation)

    # --------------------------------------------
    im1 = display_image_animation(frame, axes[0, 0], "Image")
    im2 = display_image_animation(prediction, axes[0, 1], "Prediction")
    im3 = display_overlap_mask_animation(frame, prediction, axes[0, 2], "Overlayed Mask")

    # --------------------------------------------
    x1, y1, x2, y2 = masks_to_boxes(prediction_base)[0].int()
    frame_2 = crop(frame_base, x1, y1, x2, y2, pad=pad)
    frame_2 = pad_central(frame_2, *image_pad_central)
    frame_2 = TF.resize(frame_2, size=image_size_crop, interpolation=interpolation)
    im4 = display_image_animation(frame_2, axes[1, 0], "Crop")

    # --------------------------------------------
    x1, y1, x2, y2 = masks_to_boxes(prediction_base)[0].int()
    x1, y1, x2, y2 = ema_stabilizer_crop.stabilize(x1, y1, x2, y2)
    frame_2 = crop(frame_base, x1, y1, x2, y2, pad=pad)
    frame_2 = pad_central(frame_2, *image_pad_central)
    frame_2 = TF.resize(frame_2, size=image_size_crop, interpolation=interpolation)
    im5 = display_image_animation(frame_2, axes[1, 1], "ema-stabilized")

    # --------------------------------------------
    x1, y1, x2, y2 = masks_to_boxes(prediction_base)[0].int()
    x1, y1, x2, y2 = ema_stabilizer_crop_2.stabilize(x1, y1, x2, y2)
    frame_2 = crop(frame_base, x1, y1, x2, y2, pad=pad)
    frame_2 = pad_central(frame_2, *image_pad_central)
    frame_2 = TF.resize(frame_2, size=image_size_crop, interpolation=interpolation)
    im6 = display_image_animation(frame_2, axes[1, 2], "ema-stabilized")

    # --------------------------------------------
    angle = find_angle(prediction_base)
    frame_2 = TF.rotate(frame_base, angle=angle, interpolation=T.InterpolationMode.BILINEAR)
    prediction_2 = TF.rotate(prediction_base, angle=angle, interpolation=T.InterpolationMode.NEAREST)

    x1, y1, x2, y2 = masks_to_boxes(prediction_2)[0].int()
    frame_2 = crop(frame_2, x1, y1, x2, y2, pad=pad)
    frame_2 = pad_central(frame_2, *image_pad_central)
    frame_2 = TF.resize(frame_2, size=image_size_crop, interpolation=interpolation)
    im7 = display_image_animation(frame_2, axes[2, 0], "Rotate-crop")

    # --------------------------------------------
    angle = find_angle(prediction_base)
    angle = ema_stabilizer_rotate.stabilize(angle)
    frame_2 = TF.rotate(frame_base, angle=angle, interpolation=T.InterpolationMode.BILINEAR)
    prediction_2 = TF.rotate(prediction_base, angle=angle, interpolation=T.InterpolationMode.NEAREST)

    x1, y1, x2, y2 = masks_to_boxes(prediction_2)[0].int()
    x1, y1, x2, y2 = ema_stabilizer_rotate_crop.stabilize(x1, y1, x2, y2)
    frame_2 = crop(frame_2, x1, y1, x2, y2, pad=pad)
    frame_2 = pad_central(frame_2, *image_pad_central)
    frame_2 = TF.resize(frame_2, size=image_size_crop, interpolation=interpolation)
    im8 = display_image_animation(frame_2, axes[2, 1], "ema-stabilized")

    # --------------------------------------------
    angle = find_angle(prediction_base)
    angle = ema_stabilizer_rotate_2.stabilize(angle)
    frame_2 = TF.rotate(frame_base, angle=angle, interpolation=T.InterpolationMode.BILINEAR)
    prediction_2 = TF.rotate(prediction_base, angle=angle, interpolation=T.InterpolationMode.NEAREST)

    x1, y1, x2, y2 = masks_to_boxes(prediction_2)[0].int()
    x1, y1, x2, y2 = ema_stabilizer_rotate_crop_2.stabilize(x1, y1, x2, y2)
    frame_2 = crop(frame_2, x1, y1, x2, y2, pad=pad)
    frame_2 = pad_central(frame_2, *image_pad_central)
    frame_2 = TF.resize(frame_2, size=image_size_crop, interpolation=interpolation)
    im9 = display_image_animation(frame_2, axes[2, 2], "ema-stabilized")

    # --------------------------------------------
    frames.append([im1, im2, im3, im4, im5, im6, im7, im8, im9])

ani = animation.ArtistAnimation(fig, frames, interval=10, blit=True, repeat_delay=1000)
plt.tight_layout()
plt.show()

In [ ]:
# First, close any open figures
plt.close("all")

# Then, restart the ipympl backend by rerunning the magic command
%matplotlib ipympl